In [ ]:
SELECT
    ae.id AS activity_entry_id,
    CONCAT('WIP', ae.id) AS session_id,
    ah.id AS activity_header_id,
    ah.file_number AS care_epi_number,
    CONCAT('WIP', ah.file_number) AS session_care_epi_id,
    ae.activity_date_time,
    TO_DATE(ae.activity_date_time) AS activity_date,
    ae.activity_service_id,
    srv.description AS service_description,
    acts.description AS activity_status,
    atyp.description AS activity_type,
    ae.created_date_time,
    ae.updated_date_time
FROM silver_wip_activityentry ae
LEFT JOIN silver_wip_activityheader ah
    ON ae.activity_header_id = ah.id
LEFT JOIN silver_wip_activityservice actserv
    ON actserv.id = ae.activity_service_id
LEFT JOIN silver_wip_service srv
    ON actserv.service_id = srv.id
LEFT JOIN silver_wip_activitystatus acts
    ON acts.id = ah.activity_status_id
LEFT JOIN silver_wip_activitytype atyp
    ON ae.activity_type_id = atyp.id
WHERE TRIM(CAST(ah.file_number AS STRING)) = '537633'
ORDER BY ae.activity_date_time;

In [ ]:
SELECT
    TO_DATE(ae.activity_date_time) AS activity_date,
    COUNT(*) AS row_count
FROM silver_wip_activityentry ae
LEFT JOIN silver_wip_activityheader ah
    ON ae.activity_header_id = ah.id
WHERE TRIM(CAST(ah.file_number AS STRING)) = '537633'
GROUP BY TO_DATE(ae.activity_date_time)
ORDER BY activity_date;

In [ ]:
SELECT ae.id, COUNT(*) 
FROM silver_wip_activityentry ae
LEFT JOIN silver_wip_activityentrydetail aed
    ON aed.activity_entry_id = ae.id
GROUP BY ae.id
HAVING COUNT(*) > 1;

In [ ]:
For care episode 537633, I checked the WIP silver source tables and found 6 activity entries in silver_wip_activityentry. The 3 extra dates mentioned in UAT, 27/05/2025, 10/06/2025 and 24/06/2025, are already present in the source table, each with a separate activity_entry_id. This does not appear to be caused by session_date_confirmed derivation or join duplication.

Could you please confirm whether these activities exist in the legacy WIP source, or whether they could be stale/appended records in our ingested WIP tables?

In [ ]:
Investigated session_date_confirmed for care episode 537633. The date derivation is working correctly and matches ae.activity_date_time. The extra sessions are already present in silver_wip_activityentry as separate activity entries, so this does not appear to be caused by session_date_confirmed logic or join duplication. Pending confirmation from source/legacy side on whether these are stale/appended WIP records.